# 风险预算模型

风险预算模型通过将组合风险按一定预算分配到各个资产上，实现风险的分散化配置。当所有资产的风险预算相等时，即为经典的**风险平价（Risk Parity）**模型。

> **前置阅读**：组合优化模块的整体架构、优化目标与约束条件的基类设计、CVXPC 构造器的使用方法，请先参阅 **[基本框架](基本框架.ipynb)**。

## 数学基础

### 风险度量

对于组合 $\mathbf{w}$ 的风险度量 $\mathcal{R}(\mathbf{w})$ 采用波动率度量：

$$
\mathcal{R}(\mathbf{w}) = \sigma(\mathbf{w}) = \sqrt{\mathbf{w}^T\Sigma\mathbf{w}}
$$

其中 $\Sigma$ 是协方差矩阵。边际风险贡献定义为 $\sigma(\mathbf{w})$ 相对于 $\mathbf{w}$ 的梯度：

$$
\nabla_{\mathbf{w}}\sigma(\mathbf{w}) = \frac{\Sigma\mathbf{w}}{\sqrt{\mathbf{w}^T\Sigma\mathbf{w}}}
$$

第 $i$ 个证券对于组合的风险贡献度定义为：

$$
\mathcal{RC}_i = w_i \cdot \frac{(\Sigma\mathbf{w})_i}{\sqrt{\mathbf{w}^T\Sigma\mathbf{w}}}
$$

可以将组合的总风险分解到各个证券上：

$$
\begin{aligned}
\sigma(\mathbf{w}) &= \mathbf{w}^T \cdot \frac{\Sigma\mathbf{w}}{\sqrt{\mathbf{w}^T\Sigma\mathbf{w}}} \\
&= \sum_{i=1}^{n} w_i \cdot \frac{(\Sigma\mathbf{w})_i}{\sqrt{\mathbf{w}^T\Sigma\mathbf{w}}} \\
&= \sum_{i=1}^{n} \mathcal{RC}_i
\end{aligned}
$$

### 风险预算组合

对于给定的风险预算 $\mathbf{b}$，定义风险预算组合为：

$$
\left\{
\begin{aligned}
   & \mathcal{RC}_i = b_i \cdot \mathcal{R}(\mathbf{w}) \\
   & b_i \ge 0 \\
   & w_i \ge 0 \\
   & \sum_{i=1}^{n} b_i = 1 \\
   & \sum_{i=1}^{n} w_i = 1
\end{aligned}
\right.
$$

当 $b_i = \frac{1}{n}, i=1,\ldots,n$ 时即为**风险平价模型**，即各个资产的风险贡献度相同。

## RiskBudgetObjective API 参考

### 构造方法

```python
RiskBudgetObjective(
    mask: NDArray[np.bool],                              # 股票池，True 表示可选
    budget: Optional[NDArray[np.float64]]=None,          # 风险预算，None 表示等风险预算
    factor_cov: Optional[NDArray[np.float64]]=None,      # 因子协方差阵 (k×k)
    factor_data: Optional[NDArray[np.float64]]=None,     # 因子暴露矩阵 (n×k)
    specific_risk: Optional[NDArray[np.float64]]=None,   # 特异性风险 (n,)
    cov: Optional[NDArray[np.float64]]=None,             # 证券协方差阵 (n×n)
    args: dict={},                                        # 参数设置
    config_file: Optional[str]=None                       # 配置文件路径
)
```

### 参数

无额外 `args` 参数。

- `budget=None`：自动设置为等风险预算 $\frac{1}{n}$，即**风险平价**
- `budget` 为自定义数组：指定各资产的风险预算比例，各元素之和应为 1

## 求解方法

CVXPC 使用**凸规划方法**求解风险预算问题。具体而言，将风险预算问题转化为如下凸规划：

$$
\begin{aligned}
& \underset{\mathbf{x}}{\mathop{\min}}\,\mathcal{R}(\mathbf{x}) = \sqrt{\mathbf{x}^T\Sigma\mathbf{x}} \\
& s.t.\ \sum_{i=1}^{n} b_i \ln x_i \ge c, \quad \mathbf{0} \le \mathbf{x} \le \mathbf{1}
\end{aligned}
$$

其中 $c$ 是满足 $c < \sum_{i=1}^{n} b_i \ln b_i$ 的任意常数。最后将最优解归一化得到原问题的解：

$$
\mathbf{w} = \frac{\mathbf{x}}{\mathbf{1}^T\mathbf{x}}
$$

该凸规划问题涉及**指数锥约束**，需要支持指数锥的求解器（如 CLARABEL）。

## 示例：风险平价

当不指定 `budget` 参数时，默认使用等风险预算，即风险平价模型。

In [ ]:
import numpy as np
import cvxpy as cvx

# DEMO 数据
np.random.seed(0)
nID = 10
Mask = np.full(shape=(nID,), fill_value=True, dtype=np.bool)
Cov = np.cov(np.random.randn(100 * nID, nID), rowvar=False)

In [ ]:
# 风险平价（等风险预算）
from QuantStudio.PortfolioConstructor.CVXPC import CVXPC
from QuantStudio.PortfolioConstructor.BasePC import RiskBudgetObjective, BudgetConstraint, WeightConstraint

Objective = RiskBudgetObjective(mask=Mask, cov=Cov)  # budget=None → 等风险预算
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.CLARABEL, "verbose": True}})
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))

# 验证风险贡献
total_risk = np.sqrt(Portfolio @ Cov @ Portfolio)
marginal_risk = Cov @ Portfolio / total_risk
risk_contributions = Portfolio * marginal_risk
print(f"总风险: {total_risk:.4f}")
print(f"风险贡献度: {risk_contributions / total_risk}")
print(f"求解状态: {Info['msg']}")

## 示例：自定义风险预算

可以为不同资产设置不同的风险预算。比如给低波动资产分配 60% 的风险预算，高波动资产分配 40%。

In [ ]:
# 自定义风险预算：按波动率倒数加权作为预算参考
volatility = np.sqrt(np.diag(Cov))
inv_vol_budget = 1 / volatility
custom_budget = inv_vol_budget / np.sum(inv_vol_budget)

Objective = RiskBudgetObjective(mask=Mask, budget=custom_budget, cov=Cov)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.CLARABEL}})
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))

# 验证风险预算匹配
total_risk = np.sqrt(Portfolio @ Cov @ Portfolio)
marginal_risk = Cov @ Portfolio / total_risk
actual_rc = Portfolio * marginal_risk / total_risk
print(f"目标风险预算: {custom_budget}")
print(f"实际风险贡献: {actual_rc}")
print(f"求解状态: {Info['msg']}")

## 示例：使用因子模型的风险平价

与均值方差模型一样，风险预算模型也支持因子模型输入协方差。

In [ ]:
# 模拟因子模型数据
k = 5
factor_cov = np.cov(np.random.randn(100, k), rowvar=False)
factor_data = np.random.randn(nID, k)
specific_risk = np.abs(np.random.randn(nID)) * 0.1

Objective = RiskBudgetObjective(
    mask=Mask,
    factor_cov=factor_cov, factor_data=factor_data, specific_risk=specific_risk
)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.CLARABEL}})
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(f"求解状态: {Info['msg']}")

## 示例：带约束的风险预算

风险预算模型同样可以配合各种约束条件使用。下面的例子加入了波动率约束和个股权重上限。

In [ ]:
from QuantStudio.PortfolioConstructor.BasePC import VolatilityConstraint

Objective = RiskBudgetObjective(mask=Mask, cov=Cov)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    WeightConstraint(mask=Mask, up_limit=0.15),                    # 个股权重上限 15%
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1}),
    VolatilityConstraint(mask=Mask, cov=Cov, args={"UpLimit": 0.15}),  # 波动率上限 15%
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList,
           args={"OptimOption": {"solver": cvx.CLARABEL}})
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))
print(f"组合波动率: {np.sqrt(Portfolio @ Cov @ Portfolio):.4f}")
print(f"最大个股权重: {np.max(Portfolio):.4f}")
print(f"求解状态: {Info['msg']}")

## MaxDiversificationObjective — 最大分散化组合

### 数学形式

最大分散化模型的目标是最大化组合的**分散化比率**（Diversification Ratio）：

$$
\underset{\mathbf{w}}{\mathop{\max}}\,\frac{\mathbf{w}^T\boldsymbol{\sigma}}{\sqrt{\mathbf{w}^T\Sigma\mathbf{w}}}
$$

其中 $\boldsymbol{\sigma}$ 是各证券的波动率向量（$\Sigma$ 对角元的平方根）。

分散化比率可以理解为"加权平均波动率 / 组合波动率"，越大说明分散化效果越好。

### 求解方法

CVXPC 通过两步变换求解最大分散化问题：
1. 变量替换：$\mathbf{y} = D^{-1}\mathbf{w}$，其中 $D = \operatorname{diag}(\sigma_1, \ldots, \sigma_n)$
2. 求解带约束的最小方差问题 $\min \mathbf{y}^T P \mathbf{y}$（其中 $P = D\Sigma D$ 为相关系数矩阵）
3. 反向变换：$\mathbf{w} = D\mathbf{y}$，归一化后得到最终权重

### 构造方法

```python
MaxDiversificationObjective(
    mask,                # array(shape=(n,)), 股票池
    factor_cov=None,     # 因子协方差阵
    factor_data=None,    # 因子暴露矩阵
    specific_risk=None,  # 特异性风险
    cov=None,            # 证券协方差阵
    args={},
    config_file=None
)
```

无额外 `args` 参数。

## 示例：最大分散化组合

In [ ]:
from QuantStudio.PortfolioConstructor.CVXPC import CVXPC
from QuantStudio.PortfolioConstructor.BasePC import MaxDiversificationObjective, BudgetConstraint, WeightConstraint

Objective = MaxDiversificationObjective(mask=Mask, cov=Cov)
ConstraintList = [
    WeightConstraint(mask=Mask, up_limit=1, down_limit=0),
    BudgetConstraint(mask=Mask, args={"UpLimit": 1, "DownLimit": 1})
]

PC = CVXPC(mask=Mask, objective=Objective, constraints=ConstraintList)
Portfolio, Info = PC.solve()

print("最优权重:", np.where(np.abs(Portfolio) > 1e-8, Portfolio, 0))

# 计算分散化比率
vol = np.sqrt(np.diag(Cov))
diversification_ratio = (Portfolio @ vol) / np.sqrt(Portfolio @ Cov @ Portfolio)
print(f"分散化比率: {diversification_ratio:.4f}")
print(f"组合波动率: {np.sqrt(Portfolio @ Cov @ Portfolio):.4f}")
print(f"求解状态: {Info['msg']}")

## 风险预算模型的理论性质

Maillard、Roncalli 和 Teïletche（2008）的研究详细讨论了风险平价的性质：

1. **波动率惩罚**：组合中波动较高的证券（或者相关性高的证券）在权重计算时会受到惩罚，获得更小的权重。
2. **最优性条件**：当所有成分的相关系数相同并且 Sharpe 比率也相等时，风险平价组合是 Markowitz 最优的。
3. **波动率倒数加权**：当组合所有成分证券相关系数相等时，风险平价即为波动率倒数加权。
4. **风险排序**：风险平价介于等权重和最小方差之间——其波动大于最小方差，小于等权重组合。

### 模型对比

| 模型 | 优化目标 | 分散化程度 | 参数敏感度 | 求解难度 |
|------|----------|------------|------------|----------|
| 等权重 | 无优化 | 最高（权重维度） | 无参数 | 无需求解 |
| 风险平价 | 均衡风险贡献 | 高（风险维度） | 需要协方差阵 | 凸规划（指数锥） |
| 最大分散化 | 最大化分散化比率 | 较高 | 需要协方差阵 | 二次规划 |
| 最小方差 | 最小化组合波动 | 低（可能集中） | 需要协方差阵 | 二次规划 |

风险平价在实务中因其稳健性和分散化效果而广受欢迎，尤其是在缺乏可靠预期收益预测的场景下。